In [ ]:
# Se cargan mediciones numéricas disponibles antes de priorizar una evaluación adicional.

import pandas as pd

dataframe = pd.read_csv("../data/wisc_bc_data.csv")
dataframe.shape

In [ ]:
# ¿Cuál es la clase que requiere priorización y cuántos casos representa?

target = (dataframe["diagnosis"] == "M").astype(int)
target.value_counts(normalize=True)

In [ ]:
# Se conservan dos mediciones para hacer visible la especificación del modelo.

base_features = ["texture_mean", "compactness_mean"]
X_base = dataframe[base_features].copy()
X_base.describe()

In [ ]:
# Se reserva una muestra estratificada que no participa en el ajuste.

from sklearn.model_selection import train_test_split

train_index, test_index = train_test_split(
    dataframe.index,
    train_size=0.8,
    random_state=0,
    stratify=target,
)
len(train_index), len(test_index)

In [ ]:
# Se ajusta una logística base y se expresa el resultado como probabilidad.

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

base_estimator = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=0)),
    ]
)
base_estimator.fit(X_base.loc[train_index], target.loc[train_index])
base_probability = base_estimator.predict_proba(X_base.loc[test_index])[:, 1]
base_probability[:10]

In [ ]:
# ¿Cambia el riesgo cuando la textura crece en combinación con la compactación?

X_flexible = X_base.copy()
X_flexible["texture_mean_squared"] = X_flexible["texture_mean"] ** 2
X_flexible["texture_mean_x_compactness_mean"] = (
    X_flexible["texture_mean"] * X_flexible["compactness_mean"]
)
X_flexible.head()

In [ ]:
# Se ajusta una logística cuya frontera sigue siendo lineal en los términos construidos.

flexible_estimator = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=0)),
    ]
)
flexible_estimator.fit(X_flexible.loc[train_index], target.loc[train_index])
flexible_probability = flexible_estimator.predict_proba(
    X_flexible.loc[test_index]
)[:, 1]
flexible_probability[:10]

In [ ]:
# ¿Cuál especificación ordena mejor los casos reservados sin confundir probabilidad con diagnóstico?

from sklearn.metrics import accuracy_score, roc_auc_score

model_comparison = pd.DataFrame(
    [
        {
            "model": "logistica_base",
            "test_auc": roc_auc_score(target.loc[test_index], base_probability),
            "test_accuracy": accuracy_score(
                target.loc[test_index],
                base_estimator.predict(X_base.loc[test_index]),
            ),
        },
        {
            "model": "logistica_flexible",
            "test_auc": roc_auc_score(target.loc[test_index], flexible_probability),
            "test_accuracy": accuracy_score(
                target.loc[test_index],
                flexible_estimator.predict(X_flexible.loc[test_index]),
            ),
        },
    ]
)
model_comparison

In [ ]:
# Se conservan los modelos, la comparación y sus metadatos para verificación posterior.

import json
import pickle
from pathlib import Path

submission_dir = Path("../submission")
model_comparison.to_csv(submission_dir / "model_comparison.csv", index=False)
with (submission_dir / "base_estimator.pkl").open("wb") as file:
    pickle.dump(base_estimator, file)
with (submission_dir / "flexible_estimator.pkl").open("wb") as file:
    pickle.dump(flexible_estimator, file)
with (submission_dir / "metrics.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "test_size": len(test_index),
            "positive_class": "M",
            "base_features": base_features,
            "flexible_features": list(X_flexible.columns),
            "test_auc": model_comparison.loc[1, "test_auc"],
        },
        file,
        indent=2,
    )